# 🎨 SDXL-Turbo Fast Image Studio (Colab T4 Free)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SudhirDevOps1/t4-gpu-llm-api/blob/main/colab/03_qwen_image_gen_studio.ipynb)
[![Notebook Store](https://img.shields.io/badge/Notebook_Store-Obsidian_Hub-10b981?style=for-the-badge&logo=github)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

> **What this notebook actually is:** [`stabilityai/sdxl-turbo`](https://huggingface.co/stabilityai/sdxl-turbo)
> in FP16 — 1 to 4 step text-to-image. It is **not** Qwen-Image and it does **not** use INT8/FlashAttention-2
> (T4 is Turing / SM 7.5; FlashAttention-2 needs Ampere or newer).

- ⚠️ **License:** SDXL-Turbo ships under the *Stability AI Non-Commercial Research Community License*.
  Check the license before any commercial use.
- **VRAM:** ~7 GB FP16 at 512×512 · **Speed:** 1–4 steps
- The Gradio link is password protected (Basic auth).


In [ ]:
#@title 🚀 1. Hardware Check & Session Fingerprint
import hashlib, os, socket, time, uuid

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

raw_fp = f"colab-vision-{socket.gethostname()}-{uuid.uuid4()}-{time.time()}"
INSTANCE_ID = "colab-vision-" + hashlib.sha256(raw_fp.encode()).hexdigest()[:8]
print(f"🆔 Unique Instance Fingerprint: {INSTANCE_ID}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv


In [ ]:
#@title 📦 2. Install Diffusion Dependencies
!pip install -q --upgrade diffusers transformers accelerate safetensors gradio
print("✅ Diffusers stack installed.")


In [ ]:
#@title 🎨 3. Launch SDXL-Turbo Studio (password protected)
import torch
import gradio as gr
from diffusers import AutoPipelineForText2Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"⚡ Loading stabilityai/sdxl-turbo on {DEVICE} ({DTYPE})...")
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=DTYPE,
    variant="fp16" if DEVICE == "cuda" else None,
)
pipe = pipe.to(DEVICE)
pipe.set_progress_bar_config(disable=True)


def generate_image(prompt: str, steps: int, guidance: float, width: int, height: int, seed: int):
    if not prompt.strip():
        raise gr.Error("Please enter a prompt.")
    generator = torch.Generator(device="cpu").manual_seed(int(seed))
    image = pipe(
        prompt=prompt,
        num_inference_steps=int(steps),
        guidance_scale=float(guidance),
        width=int(width),
        height=int(height),
        generator=generator,
    ).images[0]
    return image


def studio_auth():
    """Password for the public Gradio link (Basic auth).

    Priority: STUDIO_PASSWORD -> Colab secret -> Kaggle secret -> random one-time password.
    """
    import os
    import secrets

    password = os.getenv("STUDIO_PASSWORD", "")
    if not password:
        try:
            from google.colab import userdata
            password = userdata.get("STUDIO_PASSWORD") or ""
        except Exception:
            password = ""
    if not password:
        try:
            from kaggle_secrets import UserSecretsClient
            password = UserSecretsClient().get_secret("STUDIO_PASSWORD") or ""
        except Exception:
            password = ""
    if not password:
        password = secrets.token_urlsafe(9)
        print(f"\n🔑 One-time password for this session: {password}\n")
    return ("admin", password)

demo = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(label="Prompt", lines=3,
                   value="A cybernetic robot coding in a dark workstation, neon green and cyan circuits, cinematic lighting"),
        gr.Slider(minimum=1, maximum=8, value=2, step=1, label="Steps (1–4 for turbo)"),
        gr.Slider(minimum=0.0, maximum=2.0, value=0.0, step=0.1, label="Guidance (0.0 is correct for SDXL-Turbo)"),
        gr.Slider(minimum=512, maximum=1024, value=512, step=64, label="Width"),
        gr.Slider(minimum=512, maximum=1024, value=512, step=64, label="Height"),
        gr.Number(value=42, label="Seed"),
    ],
    outputs=gr.Image(label="Generated image", type="pil"),
    title="🎨 SDXL-Turbo Studio — Colab Free T4",
    description="FP16 SDXL-Turbo, 1–4 inference steps. Non-commercial license — check before commercial use.",
)

demo.launch(share=True, auth=studio_auth(), debug=False)
